In [17]:
from CNN_NAS.ChildCNNModel import ChildCNNModel
# Some magic so that the notebook will reload the external python script file any time you edit and save the .py file;
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import torch
import torch.nn as nn
import time
from torch.utils.data import DataLoader
import os

import utils

import logging
logging.basicConfig(level=logging.INFO, filename=os.path.join(os.getcwd(), 'log.log'), filemode='w')

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

device = utils.get_device_available()
print(torch.__version__)
print(device)

2.4.1
cuda


In [3]:
def is_valid_encoding(encoding):
    if len(encoding) < 3 or encoding[0] != "START" or encoding[-1] != "END":
        return False

    for i in range(1, len(encoding) - 1):
        if not encoding[i].isnumeric():
            return False

    return True


def split_dataset(data, labels, split_ratio=0.8):
    dataset = torch.utils.data.TensorDataset(data, labels)
    train_size = int(split_ratio * len(dataset))
    test_size = len(dataset) - train_size

    train_set, test_set = torch.utils.data.random_split(dataset, [train_size, test_size])

    train_data, train_labels = zip(*train_set)
    train_data = torch.stack(train_data)
    train_labels = torch.stack(train_labels)

    test_data, test_labels = zip(*test_set)
    test_data = torch.stack(test_data)
    test_labels = torch.stack(test_labels)

    return (train_data, train_labels), (test_data, test_labels)

## CIFAR

In [5]:
dataset="cifar100"

data_path = utils.check_cifar100_dataset_exists()

dataset_train_data,dataset_train_label = (torch.load(data_path + f'{dataset}/train_data.pt', weights_only=True), torch.load(data_path + f'{dataset}/train_label.pt', weights_only=True))

dataset_test_data,dataset_test_label = (torch.load(data_path + f'{dataset}/test_data.pt', weights_only=True), torch.load(data_path + f'{dataset}/test_label.pt', weights_only=True))


num_channels = 1

if len(dataset_train_data.size())==4:
    num_channels=dataset_train_data.size(1)



num_classes = dataset_train_label.unique().size(0)
height = dataset_train_data.size(-2)
width = dataset_train_data.size(-1)

print(f"Height: {height}")
print(f"Width: {width}")
print(f"Number of channels: {num_channels}")
print(f"Number of classes:  {num_classes}")



Height: 32
Width: 32
Number of channels: 3
Number of classes:  100


## Load  predefined model encoding

In [16]:
import predefined_models
# Defined by data

base_model_encoding_dict = {}

base_model_encoding_dict["Benchmark_Model"] = predefined_models.get_benchmarkModel(input_channels=num_channels, output_dim=num_classes)
base_model_encoding_dict["Lenet"] = predefined_models.get_lenet(input_channels=num_channels, output_dim=num_classes)
base_model_encoding_dict["VGG_11"] = predefined_models.get_vgg11(input_channels=num_channels, output_dim=num_classes)
base_model_encoding_dict["Alexnet"] = predefined_models.get_alexnet(input_channels=num_channels, output_dim=num_classes)


# base_model = predefined_models.get_benchmarkModel(input_channels=num_channels, output_dim=num_classes)

# print(base_model)

In [7]:
for base_model in base_model_encoding_dict:
    print(base_model)
    print(ChildCNNModel(base_model_encoding_dict[base_model], num_channels,height,width, num_classes))
    

Benchmark_Model
ChildCNNModel(
  (model): Sequential(
    (0): Conv2d(3, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(128, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Flatten(start_dim=1, end_dim=-1)
    (10): Linear(in_features=512, out_features=512, bias=True)
    (11): ReLU()
    (12): Linear(in_features=512, out_features=256, bias=True)
    (13): ReLU()
    (14): Linear(in_features=256, out_features=100, bias=True)
  )
)
Lenet
ChildCNNModel(
  (model): Sequential(
    (0): Conv2d(3, 50, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPoo

In [8]:
from CNN_NAS.CNNController import CNNController

#Load and run
logger.info("###############################################")
logger.info("STARTING TRAINING")
logger.info("###############################################")


# print(model)

for base_model in base_model_encoding_dict:
    total_epochs=10
    name = base_model
    for i in range(3):
        model = ChildCNNModel(base_model_encoding_dict[base_model], num_channels,height,width, num_classes).to(device)
        loss,train_time = model.train_model(data=dataset_train_data,label=dataset_train_label,epochs=total_epochs)
        test_accuracy = model.evaluate_model(data=dataset_test_data,labels=dataset_test_label)
        print(f"Model {name} Test Accuracy: {test_accuracy} with total epochs {total_epochs} in dataset {dataset} for time {train_time}")
        total_epochs += 10

Model Benchmark_Model Test Accuracy: (0.369, 2.5435679244995115) with total epochs 10 in dataset cifar100 for time 31.212541103363037
Model Benchmark_Model Test Accuracy: (0.3894, 2.9758223724365234) with total epochs 20 in dataset cifar100 for time 61.98938798904419
Model Benchmark_Model Test Accuracy: (0.3655, 3.636335601806641) with total epochs 30 in dataset cifar100 for time 93.46045231819153
Model Lenet Test Accuracy: (0.383, 2.4469261837005614) with total epochs 10 in dataset cifar100 for time 18.16080403327942
Model Lenet Test Accuracy: (0.3933, 2.6254132413864135) with total epochs 20 in dataset cifar100 for time 36.31141233444214
Model Lenet Test Accuracy: (0.3727, 3.3523404121398928) with total epochs 30 in dataset cifar100 for time 54.90482521057129
Model VGG_11 Test Accuracy: (0.3413, 2.7413390398025514) with total epochs 10 in dataset cifar100 for time 57.70546913146973
Model VGG_11 Test Accuracy: (0.3603, 5.013461465835571) with total epochs 20 in dataset cifar100 for ti

In [27]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import utils
from torchvision.transforms import ToPILImage

# === CONFIG ===
dataset = "cifar100"
data_path = utils.check_cifar100_dataset_exists()

# === LOAD CIFAR-100 TENSOR DATA ===
train_data = torch.load(data_path + f'{dataset}/train_data.pt', weights_only=True)
train_label = torch.load(data_path + f'{dataset}/train_label.pt', weights_only=True)
test_data = torch.load(data_path + f'{dataset}/test_data.pt', weights_only=True)
test_label = torch.load(data_path + f'{dataset}/test_label.pt', weights_only=True)

# === METADATA ===
num_channels = train_data.size(1)
height = train_data.size(-2)
width = train_data.size(-1)
num_classes = train_label.unique().size(0)

print(f"Height: {height}")
print(f"Width: {width}")
print(f"Number of channels: {num_channels}")
print(f"Number of classes:  {num_classes}")

# === DATA AUGMENTATION & NORMALIZATION (CIFAR-100) ===
mean = (0.5071, 0.4865, 0.4409)
std = (0.2673, 0.2564, 0.2761)

train_transform = transforms.Compose([
    ToPILImage(),                         # Convert tensor to PIL image
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),               # Convert back to tensor
    transforms.Normalize(mean, std)
])

test_transform = transforms.Compose([
    ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# === CUSTOM DATASET WRAPPER ===
class AugmentedTensorDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return self.images.size(0)

    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]
        # Ensure input is in [0, 1] range
        img = img.float() / 255.0 if img.max() > 1 else img
        if self.transform:
            img = self.transform(img)
        return img, label

# === DATASETS & DATALOADERS ===
dataset_train = AugmentedTensorDataset(train_data, train_label, transform=train_transform)
dataset_test = AugmentedTensorDataset(test_data, test_label, transform=test_transform)

train_loader = DataLoader(dataset_train, batch_size=200, shuffle=True, num_workers=0)
test_loader = DataLoader(dataset_test, batch_size=200, shuffle=False, num_workers=0)


Height: 32
Width: 32
Number of channels: 3
Number of classes:  100


In [28]:
from CNN_NAS.CNNController import CNNController

#Load and run
logger.info("###############################################")
logger.info("STARTING TRAINING")
logger.info("###############################################")


# print(model)

for base_model in base_model_encoding_dict:
    total_epochs=10
    name = base_model
    for i in range(3):
        model = ChildCNNModel(base_model_encoding_dict[base_model], num_channels,height,width, num_classes).to(device)
        loss,train_time = model.train_model(dataloader=train_loader,epochs=total_epochs)
        test_accuracy = model.evaluate_model(dataloader=test_loader)
        print(f"Model {name} Test Accuracy: {test_accuracy} with total epochs {total_epochs} in dataset {dataset} for time {train_time}")
        total_epochs += 10

RuntimeError: DataLoader worker (pid(s) 31328) exited unexpectedly